In [1]:
import os 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
full_data = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/final_data/Full_Dataset_UKB_ADNI_OASIS3.csv")

In [3]:
ukb = full_data.iloc[:,1:]
ukb = full_data[full_data["Dataset"] == "UKB"]

train_test_dataset_ukb, unseen_data_ukb = train_test_split(ukb, train_size=0.9, stratify=ukb["Age"], random_state=42)

train_test_dataset_ukb = train_test_dataset_ukb.iloc[:,1:]
unseen_data_ukb = unseen_data_ukb.iloc[:,1:]

adni = full_data[full_data["Dataset"] == "ADNI"]
adni["Diagnosis"].value_counts()

oasis = full_data[full_data["Dataset"] == "OASIS3"] #1893
oasis["Diagnosis"].value_counts()

adni_oasis_full_data = pd.concat([adni, oasis])

In [4]:
adni_oasis_full_data = adni_oasis_full_data.iloc[:,1:]

In [6]:
adni = adni_oasis_full_data[adni_oasis_full_data["Dataset"] == "ADNI"]
adni_hc = adni[adni["Diagnosis"] == 0.0]

train_test_dataset_adni, unseen_data_adni = train_test_split(adni_hc, test_size=0.5, random_state=42)
train_test_dataset_adni = train_test_dataset_adni.iloc[:,1:]
unseen_data_adni = unseen_data_adni.iloc[:,1:]

In [7]:
oasis3 = adni_oasis_full_data[adni_oasis_full_data["Dataset"] == "OASIS3"]
oasis3_hc = oasis3[oasis3["Diagnosis"] == 0.0]

train_test_dataset_oasis3, unseen_data_oasis3 = train_test_split(oasis3_hc, test_size=0.5, random_state=42)
train_test_dataset_oasis3 = train_test_dataset_oasis3.iloc[:,1:]
unseen_data_oasis3 = unseen_data_oasis3.iloc[:,1:]

In [8]:
#now create the dataset with 0.9 UKB, 0.5 ADNI (HC) and 0.5 OASIS3 (HC)
train_test_dataset = pd.concat([train_test_dataset_ukb, train_test_dataset_adni, train_test_dataset_oasis3], axis=0)
#train_test_dataset.to_csv("Tr_te_val_dataset.csv")

In [9]:
adni = adni_oasis_full_data[adni_oasis_full_data["Dataset"] == "ADNI"]
adni_mci_ad = adni[adni["Diagnosis"] != 0.0]

oasis3 = adni_oasis_full_data[adni_oasis_full_data["Dataset"] == "OASIS3"]
oasis3_mci_ad = oasis3[oasis3["Diagnosis"] != 0.0]

unseen_data = pd.concat([unseen_data_ukb, unseen_data_adni, adni_mci_ad, unseen_data_oasis3, oasis3_mci_ad], axis=0)
unseen_data = unseen_data
#unseen_data.to_csv("unseen_data_ukb_adni_o3.csv")

In [13]:
#making datasets for BLR based on the test data and the ADNI and OASIS3 MCI and AD patients
x = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/Data/A2_unseen_data_ukb_adni_o3.csv")
blr_test_data = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/Data/BLR_test_data.csv")
diag = x[x["Diagnosis"] != 0.0]
oasis = diag[diag["Dataset"]== "OASIS3"]
adni =  diag[diag["Dataset"]== "ADNI"]

merged = pd.concat([oasis, adni])

tr_diag, te_diag = train_test_split(merged, test_size=0.5, stratify=merged["Dataset"])

svm_train, svm_test = train_test_split(blr_test_data, test_size=0.5, stratify=blr_test_data["Dataset"]) 

#all the test data from the blr is HC
svm_train["Diagnosis"] = 0.0
svm_test["Diagnosis"] = 0.0

train = pd.concat([svm_train, tr_diag], axis=0)
test = pd.concat([svm_test, te_diag], axis=0)

train = train.iloc[:, 1:]
test = test.iloc[:, 1:]

primary_cols = ['Age', 'Sex', 'Site', 'Diagnosis', 'Dataset']
remaining_cols = [col for col in train.columns if col not in primary_cols]
train = train[primary_cols + remaining_cols]
test = test[primary_cols + remaining_cols]


# train.to_csv("B2_1.csv")
# test.to_csv("B2_2.csv")
